# Qwen2.5 Scam Detector — Fixed for Rich SchemaFixes applied:- Inference (Cell 6) and JSON parser (Cell 9) now use the correct analyst system prompt matching training- `max_new_tokens` raised to 300 everywhere — the rich schema (evidence, reasoning, confidence) is longer than the old flat schema and was getting truncated at 128- Evaluation (Cell 7) now checks Scam Type match, Risk Level match, and whether Evidence/Reasoning/Confidence fields are actually populated — not just YES/NO- Error Analysis (Cell 8) filename and column names now match what Cell 7 actually saves- `TRAIN_INSTRUCTIONS` is now defined before use (was a NameError before)- JSON parser (Cell 9) now extracts confidence, evidence (with quoted phrase + label), requested_information, and reasoning steps — not just the 5 old flat fields- **Security: the hardcoded HF token from your original notebook has been removed. Revoke that token and use a fresh one.**

## CELL 1: Install

In [1]:
!pip install -q unsloth
!pip install -q transformers datasets trl accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 98.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 110.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 116.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.

## CELL 2: Load model + LoRA

In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True,)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,

)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Model loaded")
model.print_trainable_parameters()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.36G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.6.9 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Model loaded
trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


## CELL 3: Load dataset and format with chat template

In [3]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={
        "train": "train_qwen.jsonl",
        "validation": "val_qwen.jsonl"
      }
)

def formatting_func(examples):
  texts = []

  for messages in examples["messages"]:
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    texts.append(text)
  return {"text": texts}


print("Compiling conversational Qwen blocks into flat sequence text arrays...")

train_dataset_mapped = dataset["train"].map(
        formatting_func,
        batched=True,
        remove_columns=dataset["train"].column_names
)

val_dataset_mapped = dataset["validation"].map(
        formatting_func,
        batched=True,
        remove_columns=dataset["validation"].column_names
)

print(f"Dataset prepared! Train: {len(train_dataset_mapped)} samples | Val: {len(val_dataset_mapped)} samples")

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Compiling conversational Qwen blocks into flat sequence text arrays...


Map:   0%|          | 0/25200 [00:00<?, ? examples/s]

Map:   0%|          | 0/2800 [00:00<?, ? examples/s]

Dataset prepared! Train: 25200 samples | Val: 2800 samples


In [4]:
# Sanity check: verify the rich schema fields actually appear in formatted text

sample_text = train_dataset_mapped[0]["text"]
required_fields = ["Scam:", "Scam Type:", "Risk Level:", "Confidence:", "Techniques:", "Evidence:", "Information Requested:", "Reasoning:", "Recommended Action:"]
missing = [f for f in required_fields if f not in sample_text]

if missing:
  print(f"WARNING: missing fields in formatted sample: {missing}")
else:
  print("All rich schema fields present in formatted training sample.")

All rich schema fields present in formatted training sample.


## CELL 4: TrainNote: with the richer schema (longer assistant responses — evidence, reasoning, etc.), consider raising `max_steps` from 600 to 800–1000 since each example carries more learning signal per token but also takes longer to converge on format. Watch the loss curve and adjust.

In [5]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir = "outputs_qwen_scam",
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 2,
    warmup_steps = 20,
    max_steps = 800,                 # raised from 600 — richer schema needs more steps to learn structure
    learning_rate = 2e-4,
    lr_scheduler_type = "cosine",
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 10,
    report_to = "none",
    optim = "adamw_8bit",
    weight_decay = 0.01,
    seed = 3407,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
  )

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset_mapped,
    args = training_args,
)

print("Starting training (800 steps)...")
trainer_stats = trainer.train()
print(f"Done. Final loss: {trainer_stats.training_loss:.4f}")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/25200 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Starting training (800 steps)...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 25,200 | Num Epochs = 1 | Total steps = 800
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 2 x 1) = 4
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,3.577356
20,1.964227
30,0.926486
40,0.666512
50,0.502150
60,0.490604
70,0.466888
80,0.392366
90,0.383328
100,0.369857


Unsloth: Restored added_tokens_decoder metadata in outputs_qwen_scam/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_qwen_scam/checkpoint-800/tokenizer_config.json.


Done. Final loss: 0.2566


## CELL 5: Save adapter

In [6]:
model.save_pretrained("qwen_scam_final")
tokenizer.save_pretrained("qwen_scam_final")
print("Saved.")

Unsloth: Restored added_tokens_decoder metadata in qwen_scam_final/tokenizer_config.json.


Saved.


## CELL 6: Inference (single message)

In [7]:
# CELL 6: Inference — matches new rich schema + correct system prompt
import json
import re
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

SYSTEM_PROMPT = (
    "You are an expert cybercrime and fraud detection analyst. "
    "Carefully inspect each message and provide structured scam analysis."
)

TRAIN_INSTRUCTIONS = [
    "Analyze this message for scam activity.",
    "Determine whether this message is fraudulent.",
    "Identify scam indicators in the following message.",
    "Classify this message as scam or legitimate.",
    "Review the message and provide scam analysis.",
    "Perform scam detection on this content.",
]


def analyze_scam(instruction_phrase, message):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": f"{instruction_phrase}\n\nMessage:\n{message}",
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    prompt_len = inputs.input_ids.shape[1]

    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=300,  # increased — rich schema output is longer than old flat schema
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id,
    )

    result = tokenizer.decode(
        outputs[0][prompt_len:], skip_special_tokens=True
    )
    print("=" * 50)
    print(f"INPUT MESSAGE:\n{message}")
    print("=" * 50)
    print(f"ANALYSIS RESULTS:\n{result.strip()}")
    print("=" * 50)

    return result.strip()


print("Running production inference pipeline test...")

analyze_scam(
    instruction_phrase="Determine whether this message is fraudulent.",
    message=(
        "With airtel 3G watch over 90 TV channels across Entertainment, Sports, "
        "Music, News on your phone- http://apalyaps15.com/tv/TV.jsp .To activate 3G, SMS 3G to 121"
    ),
)

Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Running production inference pipeline test...
INPUT MESSAGE:
With airtel 3G watch over 90 TV channels across Entertainment, Sports, Music, News on your phone- http://apalyaps15.com/tv/TV.jsp .To activate 3G, SMS 3G to 121
ANALYSIS RESULTS:
Scam: YES

Scam Type: Tech Support Scam

Risk Level: HIGH

Confidence: 94%

Techniques:
- Fear
- Authority
- Urgency

Evidence:
- "With airtel 3G watch over 90 TV channels across Entertainme" → Fear

Information Requested:
- Remote Access
- Payment

Reasoning:
1. Technology companies do not proactively contact users about device issues via SMS.
2. Contact leads to fraudster who installs remote access software.
3. Remote access enables financial theft.

Recommended Action:
- Do not call the number provided.
- Do not allow remote access to your device.
- Real companies do not send unsolicited support alerts.


'Scam: YES\n\nScam Type: Tech Support Scam\n\nRisk Level: HIGH\n\nConfidence: 94%\n\nTechniques:\n- Fear\n- Authority\n- Urgency\n\nEvidence:\n- "With airtel 3G watch over 90 TV channels across Entertainme" → Fear\n\nInformation Requested:\n- Remote Access\n- Payment\n\nReasoning:\n1. Technology companies do not proactively contact users about device issues via SMS.\n2. Contact leads to fraudster who installs remote access software.\n3. Remote access enables financial theft.\n\nRecommended Action:\n- Do not call the number provided.\n- Do not allow remote access to your device.\n- Real companies do not send unsolicited support alerts.'

## CELL 7: Full Evaluation — rich schema metrics, not just YES/NO

In [8]:
# # CELL 7: Full Evaluation — parses entire rich schema, not just YES/NO --> for testing on 28000 examples(full train_qwen.jsonl)
# # use if you have google collab Pro for better results
# # but as i don't have it --> i'll just use the mini evaluation script as in the later cell

# import json
# import re
# import pandas as pd
# from sklearn.metrics import classification_report, confusion_matrix
# from tqdm import tqdm
# from unsloth import FastLanguageModel

# FastLanguageModel.for_inference(model)

# val_records = []
# with open("val_qwen.jsonl", "r", encoding="utf-8") as f:
#     for line in f:
#         if line.strip():
#             val_records.append(json.loads(line))


# def extract_label(text):
#     match = re.search(r"Scam:\s*(YES|NO)", text, re.IGNORECASE)
#     return match.group(1).upper() if match else "UNKNOWN"


# def extract_confidence(text):
#     match = re.search(r"Confidence:\s*(\d+)%?", text, re.IGNORECASE)
#     return int(match.group(1)) if match else None


# def extract_scam_type(text):
#     match = re.search(r"Scam Type:\s*(.+?)(?:\n|$)", text, re.IGNORECASE)
#     return match.group(1).strip() if match else None


# def extract_risk(text):
#     match = re.search(r"Risk Level:\s*(.+?)(?:\n|$)", text, re.IGNORECASE)
#     return match.group(1).strip() if match else None


# def has_evidence(text):
#     match = re.search(
#         r"Evidence:\s*\n(.*?)(?:\nInformation Requested|\nReasoning)",
#         text,
#         re.IGNORECASE | re.DOTALL,
#     )
#     if not match:
#         return False
#     block = match.group(1).strip()
#     return block.lower() != "- none" and len(block) > 0


# def has_reasoning(text):
#     match = re.search(
#         r"Reasoning:\s*\n(.*?)(?:\nRecommended Action)",
#         text,
#         re.IGNORECASE | re.DOTALL,
#     )
#     return bool(match and len(match.group(1).strip()) > 0)


# predictions = []
# print(f"Evaluating model against {len(val_records)} validation samples...")

# for item in tqdm(val_records):
#     messages = item["messages"]
#     user_turns = messages[0:2]
#     expected_content = messages[2]["content"]
#     ground_truth = "YES" if "Scam: YES" in expected_content else "NO"
#     expected_type = extract_scam_type(expected_content)
#     expected_risk = extract_risk(expected_content)

#     prompt = tokenizer.apply_chat_template(
#         user_turns, tokenize=False, add_generation_prompt=True
#     )
#     inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
#     prompt_len = inputs.input_ids.shape[1]

#     outputs = model.generate(
#         input_ids=inputs["input_ids"],
#         attention_mask=inputs["attention_mask"],
#         max_new_tokens=300,  # rich schema needs more tokens than flat schema did
#         do_sample=False,
#         use_cache=True,
#         pad_token_id=tokenizer.eos_token_id,
#     )

#     predicted_txt = tokenizer.decode(
#         outputs[0][prompt_len:], skip_special_tokens=True
#     )
#     predicted_label = extract_label(predicted_txt)
#     predicted_type = extract_scam_type(predicted_txt)
#     predicted_risk = extract_risk(predicted_txt)
#     predicted_conf = extract_confidence(predicted_txt)
#     pred_has_evidence = has_evidence(predicted_txt)
#     pred_has_reasoning = has_reasoning(predicted_txt)

#     user_content = messages[1]["content"]
#     if "Message:\n" in user_content:
#         message_raw_text = user_content.split("Message:\n")[-1]
#     else:
#         message_raw_text = user_content

#     predictions.append(
#         {
#             "ground_truth": ground_truth,
#             "predicted_label": predicted_label,
#             "expected_scam_type": expected_type,
#             "predicted_scam_type": predicted_type,
#             "expected_risk": expected_risk,
#             "predicted_risk": predicted_risk,
#             "predicted_confidence": predicted_conf,
#             "has_evidence": pred_has_evidence,
#             "has_reasoning": pred_has_reasoning,
#             "predicted_full_text": predicted_txt.strip(),
#             "message_text": message_raw_text[:300],
#         }
#     )

# df = pd.DataFrame(predictions)
# df.to_csv("scam_detector_evaluation_results.csv", index=False)

# y_true = df["ground_truth"].tolist()
# y_pred = df["predicted_label"].tolist()
# correct_matches = sum(t == p for t, p in zip(y_true, y_pred))

# print("\n" + "=" * 50)
# print("CLASSIFICATION ACCURACY REPORT (Scam YES/NO)")
# print("=" * 50)
# print(classification_report(y_true, y_pred, labels=["YES", "NO", "UNKNOWN"]))

# cm = confusion_matrix(y_true, y_pred, labels=["YES", "NO"])
# print("CONFUSION MATRIX")
# print("Predicted YES   Predicted NO")
# print(f"Actual YES     {cm[0][0]:13d}   {cm[0][1]:12d}")
# print(f"Actual NO      {cm[1][0]:13d}   {cm[1][1]:12d}")
# print(
#     f"\nAccuracy: {correct_matches}/{len(y_true)} = {(correct_matches/len(y_true))*100:.2f}%"
# )

# # NEW: schema-richness checks — does the model actually learn the structured fields?
# print("\n" + "=" * 50)
# print("SCHEMA RICHNESS CHECK (new fields)")
# print("=" * 50)

# scam_rows = df[df["ground_truth"] == "YES"]
# type_match = (
#     scam_rows["expected_scam_type"] == scam_rows["predicted_scam_type"]
# ).mean()
# risk_match = (scam_rows["expected_risk"] == scam_rows["predicted_risk"]).mean()
# evidence_rate = scam_rows["has_evidence"].mean()
# reasoning_rate = df["has_reasoning"].mean()
# conf_present_rate = df["predicted_confidence"].notna().mean()

# print(f"Scam Type match rate (on actual scams): {type_match*100:.1f}%")
# print(f"Risk Level match rate (on actual scams): {risk_match*100:.1f}%")
# print(f"Evidence field populated on scams: {evidence_rate*100:.1f}%")
# print(f"Reasoning field populated (all rows): {reasoning_rate*100:.1f}%")
# print(f"Confidence field present (all rows): {conf_present_rate*100:.1f}%")
# print(
#     "If any of these are low, the model learned YES/NO but not the structured reasoning — needs more steps or cleaner schema in training data."
# )

## CELL 8: Error Analysis

In [9]:
# # CELL 8: Error Analysis — Phase 5
# # Fixed: correct filename + correct column names (matches Cell 7's output)
# import pandas as pd

# df = pd.read_csv("scam_detector_evaluation_results.csv")
# errors = df[df["ground_truth"] != df["predicted_label"]].copy()

# fp = errors[(errors["predicted_label"] == "YES") & (errors["ground_truth"] == "NO")]
# fn = errors[(errors["predicted_label"] == "NO") & (errors["ground_truth"] == "YES")]
# unk = errors[errors["predicted_label"] == "UNKNOWN"]

# print(f"Total errors   : {len(errors)} / {len(df)}")
# print(f"False Positives: {len(fp)}  (said scam, actually legit)")
# print(f"False Negatives: {len(fn)}  (missed real scam)")
# print(f"UNKNOWN outputs: {len(unk)}  (model didn't say YES/NO)")

# print("\n--- 10 False Negatives (scams the model missed) ---")
# for _, row in fn.head(10).iterrows():
#     print(f"Message  : {row['message_text'][:200]}")
#     print(f"Predicted: {row['predicted_full_text'][:150]}")
#     print("-" * 60)

# print("\n--- 10 False Positives (legit messages flagged as scam) ---")
# for _, row in fp.head(10).iterrows():
#     print(f"Message  : {row['message_text'][:200]}")
#     print(f"Predicted: {row['predicted_full_text'][:150]}")
#     print("-" * 60)

# print("\n--- Scam Type mismatches (correct YES/NO, wrong category) ---")
# type_mismatch = df[
#     (df["ground_truth"] == "YES")
#     & (df["predicted_label"] == "YES")
#     & (df["expected_scam_type"] != df["predicted_scam_type"])
# ]
# print(f"Count: {len(type_mismatch)}")

# for _, row in type_mismatch.head(8).iterrows():
#     print(
#         f"Expected: {row['expected_scam_type']:30s} | Predicted: {row['predicted_scam_type']}"
#     )
#     print(f"Message : {row['message_text'][:150]}")
#     print("-" * 60)


In [10]:
"""
Runs evaluation on only 100 samples (not 2800), saves CSV + prints
classification report. Takes ~8 minutes on T4 instead of 2+ hours.
No session timeout risk.
"""
# Runs on 100 stratified samples instead of the full 2800 val set.

import json, re, random
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

# ── Step 1: build a fixed 100-sample benchmark from val_qwen.jsonl ───────────
val_rows = [json.loads(l) for l in open("val_qwen.jsonl") if l.strip()]

rng = random.Random(42)
yes_rows = [r for r in val_rows if "Scam: YES" in r["messages"][2]["content"]]
no_rows  = [r for r in val_rows if "Scam: NO"  in r["messages"][2]["content"]]
benchmark_rows = rng.sample(yes_rows, 50) + rng.sample(no_rows, 50)
rng.shuffle(benchmark_rows)

print(f"Mini benchmark: {len(benchmark_rows)} samples (50 scam + 50 legit)")

# ── Step 2: run inference ─────────────────────────────────────────────────────
def extract_label(text):
    m = re.search(r"Scam:\s*(YES|NO)", text, re.IGNORECASE)
    return m.group(1).upper() if m else "UNKNOWN"

def extract_scam_type(text):
    m = re.search(r"Scam Type:\s*(.+?)(?:\n|$)", text, re.IGNORECASE)
    return m.group(1).strip() if m else "Unknown"

predictions = []

for item in tqdm(benchmark_rows, desc="Evaluating"):
    messages     = item["messages"]
    user_turns   = messages[0:2]
    expected_txt = messages[2]["content"]
    ground_truth = "YES" if "Scam: YES" in expected_txt else "NO"
    expected_type= extract_scam_type(expected_txt)

    prompt = tokenizer.apply_chat_template(
        user_turns, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    prompt_len = inputs.input_ids.shape[1]

    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=150,    # shorter than full eval — enough to catch Scam: YES/NO
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id,
    )

    raw = tokenizer.decode(outputs[0][prompt_len:], skip_special_tokens=True)
    predicted_label = extract_label(raw)
    predicted_type  = extract_scam_type(raw)

    user_content = messages[1]["content"]
    msg_text = user_content.split("Message:\n")[-1] if "Message:\n" in user_content else user_content

    predictions.append({
        "ground_truth":        ground_truth,
        "predicted_label":     predicted_label,
        "expected_scam_type":  expected_type,
        "predicted_scam_type": predicted_type,
        "correct":             ground_truth == predicted_label,
        "message_text":        msg_text[:200],
        "raw_output":          raw.strip()[:300],
    })

# ── Step 3: results ──────────────────────────────────────────────────────────
df = pd.DataFrame(predictions)
df.to_csv("mini_eval_results.csv", index=False)

y_true = df["ground_truth"].tolist()
y_pred = df["predicted_label"].tolist()
correct = sum(t == p for t, p in zip(y_true, y_pred))

print("\n" + "="*55)
print("CLASSIFICATION REPORT — 100-sample benchmark")
print("="*55)
print(classification_report(y_true, y_pred, labels=["YES","NO","UNKNOWN"]))

cm = confusion_matrix(y_true, y_pred, labels=["YES","NO"])
print("CONFUSION MATRIX")
print(f"                 Predicted YES   Predicted NO")
print(f"  Actual YES     {cm[0][0]:13d}   {cm[0][1]:12d}")
print(f"  Actual NO      {cm[1][0]:13d}   {cm[1][1]:12d}")
print(f"\nAccuracy: {correct}/100 = {correct}%")
print(f"Unknown outputs: {sum(1 for p in y_pred if p=='UNKNOWN')}")

# ── Step 4: category-level accuracy ─────────────────────────────────────────
print("\n" + "="*55)
print("PER-CATEGORY ACCURACY (scam rows only)")
print("="*55)
scam_df = df[df["ground_truth"]=="YES"].copy()
cat_acc = (
    scam_df
    .groupby("expected_scam_type")
    .apply(lambda g: round(g["correct"].mean() * 100, 1))
    .reset_index(name="accuracy_%")
    .sort_values("accuracy_%")
)
print(cat_acc.to_string(index=False))

# ── Step 5: top 10 errors ─────────────────────────────────────────────────────
print("\n" + "="*55)
print("TOP ERRORS (first 10 wrong predictions)")
print("="*55)
errors = df[~df["correct"]]
fp = errors[(errors["predicted_label"]=="YES") & (errors["ground_truth"]=="NO")]
fn = errors[(errors["predicted_label"]=="NO")  & (errors["ground_truth"]=="YES")]
print(f"False Positives (legit → SCAM): {len(fp)}")
print(f"False Negatives (scam  → CLEAN): {len(fn)}")
print("\n-- False Negatives (missed scams) --")
for _, r in fn.head(5).iterrows():
    print(f"  [{r['expected_scam_type']}] {r['message_text'][:120]}")
print("\n-- False Positives (legit flagged as scam) --")
for _, r in fp.head(5).iterrows():
    print(f"  {r['message_text'][:120]}")

print("\n✅ mini_eval_results.csv saved. Download from Colab file panel.")

Mini benchmark: 100 samples (50 scam + 50 legit)


Evaluating: 100%|██████████| 100/100 [16:00<00:00,  9.60s/it]


CLASSIFICATION REPORT — 100-sample benchmark
              precision    recall  f1-score   support

         YES       1.00      1.00      1.00        50
          NO       1.00      1.00      1.00        50
     UNKNOWN       0.00      0.00      0.00         0

    accuracy                           1.00       100
   macro avg       0.67      0.67      0.67       100
weighted avg       1.00      1.00      1.00       100

CONFUSION MATRIX
                 Predicted YES   Predicted NO
  Actual YES                50              0
  Actual NO                  0             50

Accuracy: 100/100 = 100%
Unknown outputs: 0

PER-CATEGORY ACCURACY (scam rows only)
      expected_scam_type  accuracy_%
              Bank Fraud       100.0
             Crypto Scam       100.0
           Delivery Scam       100.0
         E-commerce Scam       100.0
Government Impersonation       100.0
         Investment Scam       100.0
                Job Scam       100.0
                KYC Scam       100.0



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

## CELL 9: Structured JSON output for the web app

In [11]:
# CELL 9: Structured JSON output — full rich schema parser
# ============================================================
import json
import re
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

SYSTEM_PROMPT = (
    "You are an expert cybercrime and fraud detection analyst. "
    "Carefully inspect each message and provide structured scam analysis."
)

TRAIN_INSTRUCTIONS = [
    "Analyze this message for scam activity.",
    "Determine whether this message is fraudulent.",
    "Identify scam indicators in the following message.",
    "Classify this message as scam or legitimate.",
    "Review the message and provide scam analysis.",
    "Perform scam detection on this content.",
]


def parse_list_block(text, field_name, end_markers):
    pattern = rf"{field_name}:\s*\n(.*?)(?:\n(?:{'|'.join(end_markers)}):|$)"
    m = re.search(pattern, text, re.IGNORECASE | re.DOTALL)
    if not m:
        return []
    block = m.group(1).strip()
    if block.lower() == "- none" or not block:
        return []
    return [
        line.lstrip("- ").strip()
        for line in block.split("\n")
        if line.strip().startswith("-")
    ]


def parse_evidence_block(text):
    items = parse_list_block(
        text, "Evidence", ["Information Requested", "Reasoning"]
    )
    parsed = []
    for item in items:
        m = re.match(r'"(.+?)"\s*→\s*(.+)', item)
        if m:
            parsed.append({"text": m.group(1), "label": m.group(2).strip()})
        else:
            parsed.append({"text": item, "label": "Unknown"})
    return parsed


def analyze_scam_json(message, instruction=None):
    if instruction is None:
        instruction = TRAIN_INSTRUCTIONS[0]
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": instruction + "\n\nMessage:\n" + message,
        },
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    prompt_len = inputs.input_ids.shape[1]
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=300,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    raw = tokenizer.decode(
        outputs[0][prompt_len:], skip_special_tokens=True
    ).strip()

    def extract(pattern, text, default=""):
        m = re.search(pattern, text, re.IGNORECASE | re.DOTALL)
        return m.group(1).strip() if m else default

    scam = extract(r"Scam:\s*(YES|NO)", raw)
    scam_type = extract(r"Scam Type:\s*(.+?)(?:\n|$)", raw)
    risk = extract(r"Risk Level:\s*(.+?)(?:\n|$)", raw)
    confidence = extract(r"Confidence:\s*(\d+)", raw)
    techniques = parse_list_block(raw, "Techniques", ["Evidence"])
    evidence = parse_evidence_block(raw)
    requested = parse_list_block(
        raw, "Information Requested", ["Reasoning"]
    )
    reasoning_block = extract(
        r"Reasoning:\s*\n(.*?)(?:\nRecommended Action)", raw
    )
    reasoning = [
        re.sub(r"^\d+\.\s*", "", line).strip()
        for line in reasoning_block.split("\n")
        if line.strip()
    ]
    actions = parse_list_block(raw, "Recommended Action", ["$"])

    result = {
        "scam": scam,
        "scam_type": scam_type,
        "risk_level": risk,
        "confidence": int(confidence) if confidence.isdigit() else None,
        "techniques": techniques,
        "evidence": evidence,
        "requested_information": requested,
        "reasoning": reasoning,
        "recommended_action": actions,
        "raw_output": raw,
    }
    print(json.dumps(result, indent=2, ensure_ascii=False))
    return result


print("TEST — KYC Scam:")
analyze_scam_json(
    "URGENT: Your KYC verification is pending. Failure to update will block your account. "
    "Visit: http://sbi-kyc-verify.co"
)

print("\nTEST — Legit:")
analyze_scam_json(
    "Your HDFC bank statement for May 2025 is ready. Login to NetBanking to download."
)

print("\nTEST — UPI Scam:")
analyze_scam_json(
    "Someone sent you Rs.5,000 on Google Pay. Accept payment by entering UPI PIN here: "
    "http://gpay-accept.co/receive/5000"
)


Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


TEST — KYC Scam:


Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{
  "scam": "YES",
  "scam_type": "Phishing",
  "risk_level": "HIGH",
  "confidence": 94,
  "techniques": [
    "Fear",
    "Authority",
    "Urgency"
  ],
  "evidence": [
    {
      "text": "URGENT: Your KYC verification is pending. Failure to upda",
      "label": "Fear"
    },
    {
      "text": "http://",
      "label": "Suspicious Link"
    }
  ],
  "requested_information": [
    "Login Credentials",
    "Personal Information"
  ],
  "reasoning": [
    "Impersonates legitimate institution.",
    "Creates urgency around account security.",
    "Unofficial domain to harvest credentials."
  ],
  "recommended_action": [
    "Do not click any links.",
    "Visit official website directly.",
    "Report to cybercrime.gov.in or call 1930."
  ],
  "raw_output": "Scam: YES\n\nScam Type: Phishing\n\nRisk Level: HIGH\n\nConfidence: 94%\n\nTechniques:\n- Fear\n- Authority\n- Urgency\n\nEvidence:\n- \"URGENT: Your KYC verification is pending. Failure to upda\" → Fear\n- \"http://\" → Suspici

Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{
  "scam": "YES",
  "scam_type": "Bank Fraud",
  "risk_level": "CRITICAL",
  "confidence": 98,
  "techniques": [
    "Fear",
    "Urgency",
    "Authority"
  ],
  "evidence": [
    {
      "text": "Your HDFC bank statement for May 2025 is ready. Login to N",
      "label": "Fear"
    }
  ],
  "requested_information": [
    "OTP",
    "Bank Details",
    "Payment"
  ],
  "reasoning": [
    "Fabricates banking emergency.",
    "Provides fraudster contact or unofficial link.",
    "Designed to extract money or credentials."
  ],
  "recommended_action": [
    "Call your bank's official helpline immediately.",
    "Block card if compromised.",
    "Report to cybercrime.gov.in."
  ],
  "raw_output": "Scam: YES\n\nScam Type: Bank Fraud\n\nRisk Level: CRITICAL\n\nConfidence: 98%\n\nTechniques:\n- Fear\n- Urgency\n- Authority\n\nEvidence:\n- \"Your HDFC bank statement for May 2025 is ready. Login to N\" → Fear\n\nInformation Requested:\n- OTP\n- Bank Details\n- Payment\n\nReasoning:\n1. Fabric

{'scam': 'YES',
 'scam_type': 'UPI Scam',
 'risk_level': 'CRITICAL',
 'confidence': 98,
 'techniques': ['Trust Exploitation', 'Greed', 'Urgency'],
 'evidence': [{'text': 'Someone sent you Rs.5,000 on Google Pay. Accept payment by',
   'label': 'Credential Theft'}],
 'requested_information': ['UPI PIN'],
 'reasoning': ["Entering UPI PIN or approving collect request SENDS money from victim's account.",
  'You never need to enter PIN to receive money.',
  'Fraudster impersonates legitimate platform.'],
 'recommended_action': ['You never need to enter PIN to RECEIVE money.',
  'Reject all unexpected collect requests.',
  'Report to NPCI or cybercrime.gov.in.'],
 'raw_output': 'Scam: YES\n\nScam Type: UPI Scam\n\nRisk Level: CRITICAL\n\nConfidence: 98%\n\nTechniques:\n- Trust Exploitation\n- Greed\n- Urgency\n\nEvidence:\n- "Someone sent you Rs.5,000 on Google Pay. Accept payment by" → Credential Theft\n\nInformation Requested:\n- UPI PIN\n\nReasoning:\n1. Entering UPI PIN or approving coll

## CELL 10: Merge LoRA — run only after you're happy with eval results

In [12]:
# model.save_pretrained_merged(
#     "qwen_scam_merged_16bit", tokenizer, save_method="merged_16bit"
# )

# print("Merged model saved.")

config.json:   0%|          | 0.00/757 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in qwen_scam_merged_16bit/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [01:53<00:00, 56.57s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [01:33<00:00, 46.99s/it]


Unsloth: Merge process complete. Saved to `/content/qwen_scam_merged_16bit`
Merged model saved.


## CELL 11: Push to HuggingFace**SECURITY: do not hardcode your token.** Call `login()` with no arguments — it opens a secure prompt. If running non-interactively, set `HF_TOKEN` as a Colab secret (key icon in left sidebar) and read it with `os.environ['HF_TOKEN']`, never paste it into a cell.

In [13]:
# from huggingface_hub import login

# login()  # secure prompt — do not pass a token string here

# model.push_to_hub("PrachiSandipkumar/qwen_scam_high_accuracy")
# tokenizer.push_to_hub("PrachiSandipkumar/qwen_scam_high_accuracy")

# print("Pushed to huggingface.co/PrachiSandipkumar/qwen_scam_high_accuracy")

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.


KeyboardInterrupt

